# Day 2 - Module 0: Claude API, Tools, and Agent SDK Patterns

**Objective:** build a practical baseline for Day 2 by running one task three ways: direct Claude API call, tool-enabled loop, and an Agent SDK-style workflow that emits a reusable artifact.

By the end, you should be able to explain what came from model reasoning, what came from tool output, and what came from workflow constraints.

## Relevant Claude Code Docs
Review these before starting the module:
- [Agent SDK overview](https://code.claude.com/docs/en/agent-sdk/overview)
- [Quickstart (Agent SDK)](https://code.claude.com/docs/en/agent-sdk/quickstart)
- [Use Claude Code features in the SDK](https://code.claude.com/docs/en/agent-sdk/claude-code-features)
- [Run Claude Code programmatically](https://code.claude.com/docs/en/headless)
- [Superpowers plugin page](https://www.claudepluginhub.com/plugins/obra-superpowers-2)
- [Superpowers GitHub repository](https://github.com/obra/superpowers)
- [Superpowers command/workflow docs](https://github.com/obra/superpowers#the-basic-workflow)

## 1 - The idea

The Day 2 control modules (threat model, permissions, sandboxing, bounded workflows, hooks, injection defense) are easier when runtime boundaries are visible first.

In this module, you run the same intent through three layers:
- **Layer A:** direct API call
- **Layer B:** tool-enabled API loop
- **Layer C:** Agent SDK-style orchestrator (explore -> options -> choose -> artifact)

```mermaid
flowchart LR
  U[User Prompt] --> A[Claude API]
  A --> R[Model Response]
  R -->|tool_use| T[Local Tool Handlers]
  T --> TR[tool_result]
  TR --> A
  A --> W[Workflow Layer]
  W --> O[Artifact on disk]
```

### Setup check

Run once per session. This verifies project root, import path, and API key state before examples.

In [ ]:
import json
import os
import pathlib
import re
import subprocess
import sys
from urllib import request, error

def load_env_file(env_path):
    loaded = 0
    if not env_path.exists() or not env_path.is_file():
        return loaded
    for raw in env_path.read_text(errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, val = line.split("=", 1)
        key = key.strip()
        val = val.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = val
            loaded += 1
    return loaded

os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) in ("day1", "day2") else None
root = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
proj = root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(proj, "src"))

loaded_count = load_env_file(pathlib.Path(proj) / ".env")

print("project root:", proj)
print("loaded vars from .env:", loaded_count)
print("API key present:", bool(os.getenv("ANTHROPIC_API_KEY")))
print("model override:", os.getenv("ANTHROPIC_MODEL", "(using notebook default)"))

try:
    import contoso
    print("contoso imported OK:", contoso.__name__)
except Exception as e:
    print("Could not import contoso:", e)
    print("Run `uv sync` in repo root, then restart kernel.")

In [ ]:
DEFAULT_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-5")
API_URL = "https://api.anthropic.com/v1/messages"
API_VERSION = "2023-06-01"

def anthropic_messages(messages, model=DEFAULT_MODEL, max_tokens=700, system=None, tools=None):
    key = os.getenv("ANTHROPIC_API_KEY")
    if not key:
        raise RuntimeError("ANTHROPIC_API_KEY is not set")

    payload = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages
    }
    if system:
        payload["system"] = system
    if tools:
        payload["tools"] = tools

    req = request.Request(
        API_URL,
        data=json.dumps(payload).encode("utf-8"),
        method="POST",
        headers={
            "content-type": "application/json",
            "x-api-key": key,
            "anthropic-version": API_VERSION
        }
    )

    try:
        with request.urlopen(req, timeout=60) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except error.HTTPError as e:
        body = e.read().decode("utf-8", errors="ignore")
        raise RuntimeError(f"Anthropic API HTTP {e.code}: {body}")

def text_from_content(content_blocks):
    chunks = [b.get("text", "") for b in content_blocks if b.get("type") == "text"]
    return "\n".join([c for c in chunks if c.strip()])

## 2 - Example A: baseline API call

This run has no tools. It shows direct model behavior before workflow constraints.

In [ ]:
baseline_prompt = (
    "You are helping prepare a Day 2 workshop run. Use subagents "
    "Given the ticket 'add signed, expiring webhooks', return: "
    "(1) a one-sentence summary, (2) two risks, (3) first implementation step."
)

try:
    baseline = anthropic_messages([{"role": "user", "content": baseline_prompt}])
    baseline_text = text_from_content(baseline.get("content", []))
    print("=== Baseline response ===")
    print(baseline_text or "(no text returned)")
except Exception as e:
    print("Baseline call skipped:", e)

## 3 - Example B: tool-enabled API loop

Add two tools:
- `workspace_context_tool`: read approved workspace context
- `safety_gate_tool`: flag risky patterns before final recommendation

Tool use adds capability and traceability; it does not replace policy/sandbox controls.

In [ ]:
TOOLS = [
    {
        "name": "workspace_context_tool",
        "description": "Read a small approved text file from the workshop workspace.",
        "input_schema": {
            "type": "object",
            "properties": {
                "relative_path": {"type": "string"},
                "max_chars": {"type": "integer", "minimum": 200, "maximum": 4000}
            },
            "required": ["relative_path"]
        }
    },
    {
        "name": "safety_gate_tool",
        "description": "Evaluate text for risky instruction patterns before producing final output.",
        "input_schema": {
            "type": "object",
            "properties": {
                "candidate_text": {"type": "string"}
            },
            "required": ["candidate_text"]
        }
    }
]

ALLOWED_PREFIXES = ("scenarios/", "reference/day2/", "templates/")

def workspace_context_tool(relative_path, max_chars=1600):
    rel = str(relative_path or "").strip()
    if not rel or not rel.startswith(ALLOWED_PREFIXES):
        return {"ok": False, "reason": "path not in allowed prefixes", "path": rel}

    target = pathlib.Path(proj) / rel
    if not target.exists() or not target.is_file():
        return {"ok": False, "reason": "file missing", "path": rel}

    text = target.read_text(errors="ignore")
    limit = max(200, min(int(max_chars), 4000))
    return {"ok": True, "path": rel, "preview": text[:limit], "truncated": len(text) > limit}

def safety_gate_tool(candidate_text):
    text = str(candidate_text or "")[:6000]
    patterns = [
        r"(?i)ignore\s+previous\s+instructions",
        r"(?i)exfiltrat(e|ion)|send\s+secrets|dump\s+env",
        r"(?i)curl\s+http|wget\s+http",
        r"(?i)eval\(|exec\("
    ]
    hits = [p for p in patterns if re.search(p, text)]
    return {"ok": len(hits) == 0, "risk_level": "high" if hits else "low", "matched_rules": hits}

def run_local_tool(name, tool_input):
    if name == "workspace_context_tool":
        return workspace_context_tool(tool_input.get("relative_path", ""), tool_input.get("max_chars", 1600))
    if name == "safety_gate_tool":
        return safety_gate_tool(tool_input.get("candidate_text", ""))
    return {"ok": False, "reason": f"unknown tool: {name}"}

In [ ]:
TOOL_SYSTEM = (
    "You are a workshop assistant. Use tools when needed. "
    "Always run safety_gate_tool on your final recommendation text before finalizing."
)

def run_tool_loop(user_prompt, max_rounds=5):
    messages = [{"role": "user", "content": user_prompt}]
    trace = []

    for _ in range(max_rounds):
        resp = anthropic_messages(messages, system=TOOL_SYSTEM, tools=TOOLS, max_tokens=900)
        assistant_content = resp.get("content", [])
        messages.append({"role": "assistant", "content": assistant_content})

        tool_uses = [b for b in assistant_content if b.get("type") == "tool_use"]
        text_answer = text_from_content(assistant_content)

        if not tool_uses:
            return {"final_text": text_answer, "trace": trace, "messages": messages}

        tool_results = []
        for tu in tool_uses:
            name = tu.get("name")
            tinput = tu.get("input", {})
            output = run_local_tool(name, tinput)
            trace.append({"tool": name, "input": tinput, "output": output})

            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tu.get("id"),
                "content": json.dumps(output)
            })

        messages.append({"role": "user", "content": tool_results})

    return {"final_text": "Tool loop reached max rounds before completion.", "trace": trace, "messages": messages}

tool_prompt = (
    "Read context from scenarios/d2-capstone-signed-webhooks.md, then produce: "
    "(1) one-sentence summary, (2) two implementation options, "
    "(3) one recommendation with one risk note."
)

try:
    tool_run = run_tool_loop(tool_prompt)
    print("=== Tool loop final text ===")
    print(tool_run["final_text"] or "(no final text)")
    print("\n=== Tool trace summary ===")
    for i, t in enumerate(tool_run["trace"], start=1):
        print(f"{i}. {t['tool']} -> ok={t['output'].get('ok')}")
except Exception as e:
    tool_run = {"final_text": "", "trace": []}
    print("Tool loop skipped:", e)

## 4 - Example C: Agent SDK-style examples (including superpowers workflow)

The classes below model the structure you would usually put behind an Agent SDK interface:
- minimal run wrapper
- tool-enabled run wrapper
- superpowers-style workflow agent that emits structured artifact sections

Superpowers flow for this notebook:
```mermaid
flowchart TD
    E[Explore Context] --> O[Generate Options]
    O --> C[Choose Recommendation]
    C --> A[Create Artifact]
    A --> V[Run Self Check]
    V --> B[Bridge to Threat Model]
```

What this demonstrates:
- process discipline (fixed phase order)
- explicit outputs per phase
- traceable handoff from model output to persisted artifact

In [ ]:
class MinimalAgent:
    def __init__(self, model=DEFAULT_MODEL):
        self.model = model

    def run(self, task):
        resp = anthropic_messages([{"role": "user", "content": task}], model=self.model, max_tokens=700)
        return text_from_content(resp.get("content", []))

class ToolEnabledAgent:
    def run(self, task):
        return run_tool_loop(task)

class SuperpowersWorkflowAgent:
    def __init__(self):
        self.tool_agent = ToolEnabledAgent()

    def run(self, scenario_path):
        explore = self.tool_agent.run(f"Read context from {scenario_path} and summarize in 4 bullets.")
        options = self.tool_agent.run(f"Using {scenario_path}, produce two implementation options with tradeoffs.")
        choose = self.tool_agent.run(f"Choose one recommendation for {scenario_path} with one risk and one mitigation.")

        return {
            "explore": explore.get("final_text", ""),
            "options": options.get("final_text", ""),
            "choose": choose.get("final_text", ""),
            "tool_trace_counts": {
                "explore": len(explore.get("trace", [])),
                "options": len(options.get("trace", [])),
                "choose": len(choose.get("trace", []))
            }
        }

In [ ]:
scenario_for_demo = "scenarios/d2-capstone-signed-webhooks.md"

try:
    agent_a = MinimalAgent()
    agent_b = ToolEnabledAgent()
    agent_c = SuperpowersWorkflowAgent()

    out_a = agent_a.run("In 3 bullets, explain signed and expiring webhooks.")
    out_b = agent_b.run(f"Read {scenario_for_demo} and return one recommendation with one risk note.")
    out_c = agent_c.run(scenario_for_demo)

    print("=== MinimalAgent ===")
    print(out_a or "(no output)")
    print("\n=== ToolEnabledAgent (final) ===")
    print(out_b.get("final_text", "(no output)"))
    print("tool calls:", len(out_b.get("trace", [])))
    print("\n=== SuperpowersWorkflowAgent ===")
    print("sections:", ", ".join(out_c.keys()))
    print("trace counts:", out_c.get("tool_trace_counts", {}))
except Exception as e:
    out_c = {}
    print("Agent SDK examples skipped:", e)

## 5 - Example D: plan -> analyze -> execute -> validate (Agent SDK pattern)

This example shows a more realistic orchestration shape while staying teachable:
- each phase has one explicit purpose,
- each phase emits an artifact section,
- validation is done both by a model check and a deterministic safety gate.

```mermaid
flowchart LR
  P[Plan] --> A[Analyze]
  A --> E[Execute]
  E --> V[Validate]
  V --> O[Workflow Packet]
```

Design goal: add complexity in **workflow structure**, not in code volume.

In [ ]:
class PlanAnalyzeExecuteValidateAgent:
    def __init__(self, worker=None):
        self.worker = worker or ToolEnabledAgent()

    def _phase(self, label, prompt):
        result = self.worker.run(prompt)
        return {
            "label": label,
            "text": result.get("final_text", ""),
            "tool_calls": len(result.get("trace", [])),
            "trace": result.get("trace", [])
        }

    def run(self, scenario_path):
        plan_prompt = (
            f"PLAN phase for {scenario_path}: return a concise plan with goals, assumptions, and 3 implementation steps."
        )
        plan = self._phase("plan", plan_prompt)

        analyze_prompt = (
            f"ANALYZE phase for {scenario_path}: review this plan for risks and hidden coupling. "
            f"Plan text: {plan['text']}"
        )
        analyze = self._phase("analyze", analyze_prompt)

        execute_prompt = (
            f"EXECUTE phase for {scenario_path}: draft a minimal execution packet with first actions, test intent, and rollback note. "
            f"Use analysis notes: {analyze['text']}"
        )
        execute = self._phase("execute", execute_prompt)

        validate_prompt = (
            f"VALIDATE phase for {scenario_path}: check if execution packet is complete and state pass/fail with reasons. "
            f"Execution packet: {execute['text']}"
        )
        validate = self._phase("validate", validate_prompt)

        gate = run_local_tool("safety_gate_tool", {"candidate_text": execute["text"]})

        return {
            "plan": plan,
            "analyze": analyze,
            "execute": execute,
            "validate": validate,
            "safety_gate": gate
        }

try:
    p2v_agent = PlanAnalyzeExecuteValidateAgent()
    p2v_output = p2v_agent.run(scenario_for_demo)

    print("=== plan -> analyze -> execute -> validate ===")
    for phase in ("plan", "analyze", "execute", "validate"):
        info = p2v_output[phase]
        snippet = (info["text"] or "").replace("\n", " ")[:160]
        print(f"{phase:8} | tool_calls={info['tool_calls']:2d} | {snippet}")

    print("\nSafety gate:", p2v_output["safety_gate"])
except Exception as e:
    p2v_output = {}
    print("Complex workflow example skipped:", e)

## 6 - Your exercise

Use one Day 2 scenario and produce an artifact packet via the workflow pattern:

1. Explore context from the scenario.
2. Generate two options.
3. Choose one recommendation.
4. Run safety gating over recommendation text.
5. Save output to `artifacts/day2/m0/claude-agent-sdk-lab.md`.

Suggested scenarios:
- `scenarios/d2-capstone-signed-webhooks.md`
- `scenarios/d2-m4-impossible.md`
- `scenarios/d2-m6-add-analytics-field.md`

In [ ]:
def write_module0_artifact(artifact_dict, scenario_path):
    out_dir = pathlib.Path(proj) / "artifacts" / "day2" / "m0"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / "claude-agent-sdk-lab.md"

    content = [
        "# Day 2 Module 0 - Claude API and Agent SDK Lab",
        "",
        f"Scenario: `{scenario_path}`",
        "",
        "## Explore",
        artifact_dict.get("explore", "(missing)"),
        "",
        "## Options",
        artifact_dict.get("options", "(missing)"),
        "",
        "## Chosen Recommendation",
        artifact_dict.get("choose", "(missing)"),
        "",
        "## Tool Trace Counts",
        json.dumps(artifact_dict.get("tool_trace_counts", {}), indent=2),
        ""
    ]

    out_file.write_text("\n".join(content))
    return out_file

if out_c:
    artifact_path = write_module0_artifact(out_c, scenario_for_demo)
    print("artifact written:", artifact_path)
else:
    print("No artifact to write. Run previous cells with a valid API key first.")

### Verify your work

This checklist is advisory, not a gate. It verifies that the expected Module 0 artifact exists and includes required sections.

In [ ]:
m0 = pathlib.Path(proj) / "artifacts" / "day2" / "m0" / "claude-agent-sdk-lab.md"
if m0.exists():
    t = m0.read_text().lower()
    print(f"[x] artifact present ({len(t.split())} words)")
    print(f"  [{'x' if '## explore' in t else ' '}] has Explore section?")
    print(f"  [{'x' if '## options' in t else ' '}] has Options section?")
    print(f"  [{'x' if '## chosen recommendation' in t else ' '}] has Chosen Recommendation section?")
    print(f"  [{'x' if 'tool trace counts' in t else ' '}] has Tool Trace Counts section?")
else:
    print("[ ] artifacts/day2/m0/claude-agent-sdk-lab.md missing")

## 7 - Debrief

- What changed from API-only to tool-enabled runs?
- What did the workflow layer add beyond prompt text?
- Which parts of this flow are safe by policy, which by runtime isolation, and which still require human gatekeeping?

Bridge to Module 1: now that mechanics are visible, threat modeling can focus on where stage reach expands and which control closes each path first.